# CD CosMx Resection: Early vs Late Treg Proximity to Myeloid Cells (Ext Fig 10A-C)

Chi-sq tests for Early vs Late Treg proximity to CD9+ Mac and Neutrophils.
Thresholds: 25–150px distance radii.

In [ ]:
# ── Paths ──
DATA_DIR   = "/share/fsmresfiles/UC/AtoMx/CD_surgical_resection"
OUTPUT_DIR = DATA_DIR + "/combined/treg_colocal"

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import squidpy as sq
import matplotlib.pyplot as plt
import seaborn as sns
import os
import scanpy.external as sce
from scipy import stats


In [ ]:
adata=sc.read_h5ad('/share/fsmresfiles/UC/AtoMx/CD_6k_wholetrans/whole_trans/Processed_merged/h5ad_obj/norm_anno.h5ad')

In [ ]:
adata.obs['cell_type_short'].value_counts()

In [ ]:
treg_others = []
for i,j in adata.obs.iterrows():
    cell_cat = j['cell_category']
    cell_type = j['cell_type_short']
    if cell_type == 'Treg':
        treg_others.append('Treg')
    elif cell_cat=='Myeloid':
        treg_others.append('Myeloid')
    elif cell_cat=='Connective':
        treg_others.append('Connective')
    else:
        treg_others.append('Others')

In [ ]:
adata.obs['treg_others']=treg_others

In [ ]:
adata.obs['treg_others'].value_counts()

In [ ]:
adata.obs['Treg']

In [ ]:
adata1=adata[adata.obs['orig.ident']=='CD_B4']
adata2=adata[adata.obs['orig.ident']=='CD_B5']

In [ ]:
adata_cosmx2_s1 = adata2[adata2.obs['slide']=='CD_B5_slide1']
adata_cosmx2_s3 = adata2[adata2.obs['slide']=='CD_B5_slide3']
adata_cosmx2_s5 = adata2[adata2.obs['slide']=='CD_B5_slide5']

In [ ]:
adata_cosmx1_s1 = adata1[adata1.obs['slide']=='CD_B4_slide1']
adata_cosmx1_s3 = adata1[adata1.obs['slide']=='CD_B4_slide3']
adata_cosmx1_s4 = adata1[adata1.obs['slide']=='CD_B4_slide4']

In [ ]:
tregs = adata[adata.obs['cell_type']=='Treg']

In [ ]:
tregs.X = tregs.layers['raw']

In [ ]:
tregs.X.toarray()

In [ ]:
sc.pp.normalize_total(tregs, target_sum=1e4)   # library-size normalize
#sc.pp.log1p(tregs) # freeze a log-normalized snapshot for plotting

In [ ]:
norm_counts = pd.DataFrame(tregs.X.toarray())
norm_counts.columns = tregs.var_names.tolist()
norm_counts.index = tregs.obs_names.tolist()

In [ ]:
norm_counts

In [ ]:
norm_counts.to_csv('/share/fsmresfiles/UC/AtoMx/CD_6k_wholetrans/whole_trans/Processed_merged/count_csv/treg_norm_counts.h5ad')

In [ ]:
sc.pp.highly_variable_genes(
    tregs, n_top_genes=3000, flavor='seurat_v3',layer='raw'
)

In [ ]:
#sc.pp.scale(tregs, max_value=10)


In [ ]:
sc.tl.pca(tregs, n_comps=30, use_highly_variable=True, )
# Harmony integrates on PCA embeddings; writes to X_pca_harmony
sce.pp.harmony_integrate(tregs, key='patient')  # or 'batch'

sc.pp.neighbors(tregs, use_rep='X_pca_harmony', n_neighbors=20, n_pcs=30)
sc.tl.umap(tregs)


In [ ]:
panels = {
 'Activated_tissue': ['BATF','PRDM1','MAF','TNFRSF18','TNFRSF4'],
 'Tfr': ['BCL6','CXCR5','PDCD1','ICOS'],
 'Th17_like': ['RORC','CCR6','IL17A','IL17F'],
 'IFN_response': ['ISG15','IFI6','MX1','IFIT3'],
 'Cell_cycle': ['MKI67','TOP2A','STMN1'],
 'Treg_core':['FOXP3','IL2RA','CTLA4','TIGIT','ENTPD1','LRRC32','IL10']
}
for name, genes in panels.items():
    sc.tl.score_genes(tregs, [g for g in genes if g in tregs.var_names], score_name=name)

In [ ]:
# Calculate binary prevalence score for each gene set
for name, genes in panels.items():
    # Filter genes that exist in the dataset
    valid_genes = [g for g in genes if g in tregs.var_names]
    
    if len(valid_genes) > 0:
        # Get expression matrix for these genes
        gene_expr = tregs[:, valid_genes].X
        
        # Convert to dense if sparse
        if hasattr(gene_expr, 'toarray'):
            gene_expr = gene_expr.toarray()
        
        # Binary: expressed (>0) or not (0)
        binary_expr = (gene_expr > 0).astype(int)
        
        # Calculate proportion of genes expressed per cell
        prevalence_score = binary_expr.sum(axis=1) / len(valid_genes)
        
        # Store in obs
        tregs.obs[f'{name}_prevalence'] = prevalence_score
        
        print(f"{name}: {len(valid_genes)}/{len(genes)} genes found")


In [ ]:
for name in panels.keys():
    sc.pl.umap(
        tregs,
        color=name,
        cmap="viridis",     # or 'Reds', 'coolwarm', etc.
        vmin='p1', vmax='p99',  # clip extreme values for nicer contrast
        frameon=False,size =20
    )


In [ ]:
treg_strict = tregs[tregs.obs["Treg_core"] >= 0].copy()

In [ ]:
treg_strict = treg_strict[treg_strict.obs["Cell_cycle"] < 4].copy()

In [ ]:
treg_counts = pd.DataFrame(treg_strict.obs['condition'].value_counts())
treg_counts.columns = ['Treg']

In [ ]:
treg_counts

In [ ]:
total_counts = pd.DataFrame(adata.obs['condition'].value_counts())
total_counts.columns = ['total']

In [ ]:
df = treg_counts.merge(total_counts,left_index=True, right_index=True)

In [ ]:
df['condition']=df.index.tolist()
df.reset_index(drop=True)

In [ ]:
palette_dict = {
    "Healthy": "#16a34a",   # darker green
    "Non-Inf": "#f59e0b",  # darker yellow
    "Inf": "#b91c1c"     # dark red
}


In [ ]:
# Desired order
order = ["Healthy", "Non-Inf", "Inf",]

df["condition"] = pd.Categorical(df["condition"], categories=order, ordered=True)
df = df.reset_index(drop=True)
df = df.sort_values("condition")  
df["Fraction"] = df["Treg"] / df["total"]
df["color"] = df["condition"].map(palette_dict)

# Color mapping by status
def status_from_name(name: str) -> str:
    if name.startswith("Healthy"):  return "Control"
    if name.startswith("Inf"): return "Inactive"
    if name.startswith("Non-inf"):   return "Active"

status = [status_from_name(c) for c in df["condition"]]


In [ ]:
clean_tregs =treg_strict.obs['cell'].tolist()

In [ ]:
cell_type_treg = []
for i,j in adata.obs.iterrows():
    cell_type = j['cell_type_short']
    if cell_type == 'Treg':
        if j['cell'] in clean_tregs:
            cell_type_treg.append('Core Treg')
        else:
            cell_type_treg.append('NC Treg')
    else:
        cell_type_treg.append(cell_type)

In [ ]:
adata.obs['cell_type_treg']=cell_type_treg

# pt bin label transfer

In [ ]:
pt = pd.read_csv('/share/fsmresfiles/UC/AtoMx/CD_6k_wholetrans/whole_trans/Processed_merged/anno/treg_pt_transfer_labels.csv')

In [ ]:
pt['bin']=['Late' if i == 4 else 'Early' for i in pt['predicted.id'].tolist()]

In [ ]:
treg_strict.obs['Treg_bin']=['Treg_'+str(i) for i in pt['bin'].tolist()]

In [ ]:
pt_all = pd.read_csv('/share/fsmresfiles/UC/AtoMx/CD_6k_wholetrans/whole_trans/Processed_merged/anno/alltreg_pt_transfer_labels.csv')

In [ ]:
pt_all['bin']=['Late' if i == 4 else 'Early' for i in pt_all['predicted.id'].tolist()]

In [ ]:
tregs.obs['Treg_bin']=['Treg_'+str(i) for i in pt_all['bin'].tolist()]

In [ ]:
adata.obs['treg_others'].value_counts()

In [ ]:
tregs_dict = tregs.obs.set_index('cell')['Treg_bin'].to_dict()
treg_strict_dict = treg_strict.obs.set_index('cell')['Treg_bin'].to_dict()


In [ ]:
treg_bin = []
all_treg_bin = []

for i, j in adata.obs.iterrows():
    treg_others = j['treg_others']
    cell_id = j['cell']  # Get the cell identifier
    
    if treg_others == 'Treg':
        all_treg_bin.append(tregs_dict[cell_id])
        # Check if cell exists in treg_strict
        if cell_id in treg_strict_dict:
            treg_bin.append(treg_strict_dict[cell_id])
        else:
            treg_bin.append('NC Treg')
    else:
        treg_bin.append(treg_others)
        all_treg_bin.append(treg_others)

# Add back to adata
adata.obs['strict_treg_pt'] = treg_bin
adata.obs['all_treg_pt'] = all_treg_bin

In [ ]:
adata.obs['strict_treg_pt'].value_counts()

In [ ]:
adata.obs['all_treg_pt'].value_counts()

In [ ]:
adata1=adata[adata.obs['orig.ident']=='CD_B4']
adata2=adata[adata.obs['orig.ident']=='CD_B5']

In [ ]:
adata_cosmx2_s1 = adata2[adata2.obs['slide']=='CD_B5_slide1']
adata_cosmx2_s3 = adata2[adata2.obs['slide']=='CD_B5_slide3']
adata_cosmx2_s5 = adata2[adata2.obs['slide']=='CD_B5_slide5']

In [ ]:
adata_cosmx1_s1 = adata1[adata1.obs['slide']=='CD_B4_slide1']
adata_cosmx1_s3 = adata1[adata1.obs['slide']=='CD_B4_slide3']
adata_cosmx1_s4 = adata1[adata1.obs['slide']=='CD_B4_slide4']

In [ ]:
tregs_obs = treg_strict.obs
tregs_obs

# spatial proximity strict Tregs early vs late

In [ ]:
adata_treg_mye=adata[adata.obs['cell_type_general'].isin(['SELENOP+ Mac',
 'DC',
 'Mo-Mac',
 'CD9+ M2 Mac',
 'Neutrophil',
 'Treg'])]

In [ ]:
adata_treg_mye_strict = adata_treg_mye[(adata_treg_mye.obs['cell_type_treg']=='Core Treg') | (adata_treg_mye.obs['cell_category'].isin(['Myeloid']))]

In [ ]:
adata_treg_mye_strict.obs['cell_type_treg'].value_counts()

In [ ]:
cell_type_treg_cci = []
for i,j,k,l in zip(adata_treg_mye_strict.obs['cell_type_treg'].tolist(), adata_treg_mye_strict.obs['cell_type_general'].tolist(),
                 adata_treg_mye_strict.obs['cell_category'].tolist(),adata_treg_mye_strict.obs['all_treg_pt'].tolist(), ):
    if 'DC' in i:
        cell_type_treg_cci.append ('DC')
    elif 'Mo-Mac' in i:
        cell_type_treg_cci.append ('Mo-Mac')
    elif i == ['Mac S+SG+','Mac S+XS-']:
        cell_type_treg_cci.append ('Mac S+SG+')
    elif j == 'Treg':
        cell_type_treg_cci.append (l)
    elif k == 'Connective':
        cell_type_treg_cci.append(j)
    else:
        cell_type_treg_cci.append(i)

In [ ]:
adata_treg_mye_strict.obs['cell_type_treg_cci']=cell_type_treg_cci

In [ ]:
adata_treg_mye_strict.obs['cell_type_treg_cci'].value_counts()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import squidpy as sq
from scipy.sparse import block_diag

# 1) Build per-slide graphs
Cs, Ds, order = [], [], []
for slide, idx in adata_treg_mye_strict.obs.groupby("slide").indices.items():
    ad = adata_treg_mye_strict[idx].copy()

    # (optional) rebase; translation doesn't change distances
    xy = ad.obsm["spatial_fov"].astype(float)
    ad.obsm["spatial_fov"] = xy - xy.min(axis=0, keepdims=True)

    # neighbors within slide (choose ONE of kNN or radius)
    sq.gr.spatial_neighbors(ad, spatial_key="spatial_fov", coord_type="generic", n_neighs=6)
    # e.g., radius alt: sq.gr.spatial_neighbors(ad, spatial_key="spatial_fov", coord_type="generic", radius=50.0)

    Cs.append(ad.obsp["spatial_connectivities"])
    Ds.append(ad.obsp["spatial_distances"])
    order.extend(idx.tolist())

# 2) Stitch block-diagonal graph on a reordered copy
adata_bd = adata_treg_mye_strict[order].copy()
adata_bd.obsp["spatial_connectivities"] = block_diag(Cs, format="csr")
adata_bd.obsp["spatial_distances"]      = block_diag(Ds, format="csr")

# Quick sanity check: distances should now reflect within-slide mins
d = adata_bd.obsp["spatial_distances"].data
print("Min/95th percentile distances (block-diag):", float(d.min()), float(np.percentile(d, 95)))



In [ ]:
early_treg_count = 93#208
late_treg_count = 141#286

In [ ]:
print(f"Early Treg near CD9+ Mac: {early_cd9:.2%}")
print(f"Late Treg near CD9+ Mac: {late_cd9:.2%}")
print(f"Difference: {(late_cd9 - early_cd9):.2%}")

# Statistical test
from scipy.stats import chi2_contingency

# Create contingency table
early_close = int(early_cd9 * early_treg_count)
early_far = early_treg_count - early_close
late_close = int(late_cd9 * late_treg_count)
late_far = late_treg_count - late_close

table = [[early_close, early_far], [late_close, late_far]]
chi2, p_value, dof, expected = chi2_contingency(table)
print(f"Chi-square p-value: {p_value:.4f}")

In [ ]:
print(f"Early Treg near Neut: {early_neut:.2%}")
print(f"Late Treg near Neut: {late_neut:.2%}")
print(f"Difference: {(late_neut - early_neut):.2%}")


# Create contingency table
early_close = int(early_neut * early_treg_count)
early_far = early_treg_count - early_close
late_close = int(late_neut * late_treg_count)
late_far = late_treg_count - late_close

table = [[early_close, early_far], [late_close, late_far]]
chi2, p_value, dof, expected = chi2_contingency(table)
print(f"Chi-square p-value: {p_value:.4f}")

In [ ]:
thresholds = [25, 50, 75, 100, 150]
for thresh in thresholds:
    early = calculate_proximity_score(adata_treg_mye_fib_strict, "Treg_Early", "CD9+ Mac", threshold=thresh)
    late = calculate_proximity_score(adata_treg_mye_fib_strict, "Treg_Late", "CD9+ Mac", threshold=thresh)
    print(f"Threshold {thresh}: Early={early:.2%}, Late={late:.2%}, Diff={late-early:.2%}")

In [ ]:
thresholds = [25, 50, 75, 100, 150]
for thresh in thresholds:
    early = calculate_proximity_score(adata_treg_mye_fib_strict, "Treg_Early", "Neutrophil", threshold=thresh)
    late = calculate_proximity_score(adata_treg_mye_fib_strict, "Treg_Late", "Neutrophil", threshold=thresh)
    print(f"Threshold {thresh}: Early={early:.2%}, Late={late:.2%}, Diff={late-early:.2%}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import chi2_contingency
from scipy.spatial import distance_matrix

def calculate_proximity_score(adata, celltype1, celltype2, threshold=100):
    """Calculate proximity score only"""
    mask1 = adata.obs['cell_type_treg_cci'] == celltype1
    mask2 = adata.obs['cell_type_treg_cci'] == celltype2
    
    if mask1.sum() == 0 or mask2.sum() == 0:
        return np.nan
    
    coords1 = adata[mask1].obsm['spatial']
    coords2 = adata[mask2].obsm['spatial']
    
    dist_matrix_calc = distance_matrix(coords1, coords2)
    proximity = (dist_matrix_calc.min(axis=1) < threshold).mean()
    
    return proximity

def calculate_early_vs_late_pvalue(adata, early_type, late_type, target_type, threshold=100):
    """Calculate p-value comparing Early vs Late Treg proximity to target"""
    
    # Get proximity scores
    early_mask = adata.obs['cell_type_treg_cci'] == early_type
    late_mask = adata.obs['cell_type_treg_cci'] == late_type
    target_mask = adata.obs['cell_type_treg_cci'] == target_type
    
    if early_mask.sum() == 0 or late_mask.sum() == 0 or target_mask.sum() == 0:
        return np.nan
    
    # Calculate distances for early tregs
    coords_early = adata[early_mask].obsm['spatial']
    coords_late = adata[late_mask].obsm['spatial']
    coords_target = adata[target_mask].obsm['spatial']
    
    dist_early = distance_matrix(coords_early, coords_target)
    dist_late = distance_matrix(coords_late, coords_target)
    
    # Count close vs far
    early_close = (dist_early.min(axis=1) < threshold).sum()
    early_far = len(coords_early) - early_close
    late_close = (dist_late.min(axis=1) < threshold).sum()
    late_far = len(coords_late) - late_close
    
    # Chi-square test
    contingency = [[early_close, early_far],
                   [late_close, late_far]]
    
    chi2, p_value, dof, expected = chi2_contingency(contingency)
    
    return p_value

# Collect results
thresholds = [50, 75, 100, 125, 150]
cell_pairs = [
    ("Treg_Early", "CD9+ Mac", "Early Treg", "CD9+ Mac"),
    ("Treg_Late", "CD9+ Mac", "Late Treg", "CD9+ Mac"),
    ("Treg_Early", "Neutrophil", "Early Treg", "Neutrophil"),
    ("Treg_Late", "Neutrophil", "Late Treg", "Neutrophil")
]

results = {}
for celltype1, celltype2, treg_label, target_label in cell_pairs:
    key = f"{treg_label}_{target_label}"
    results[key] = {'thresholds': [], 'proximity': [], 'pvalues': []}
    
    for thresh in thresholds:
        prox = calculate_proximity_score(
            adata_treg_mye_fib_strict, celltype1, celltype2, threshold=thresh
        )
        results[key]['thresholds'].append(thresh)
        results[key]['proximity'].append(prox)

# Calculate Early vs Late p-values for each target and threshold
early_vs_late_pvals = {}
for target_label in ['CD9+ Mac', 'Neutrophil']:
    early_vs_late_pvals[target_label] = []
    target_type = target_label
    
    for thresh in thresholds:
        pval = calculate_early_vs_late_pvalue(
            adata_treg_mye_fib_strict, 
            "Treg_Early", 
            "Treg_Late", 
            target_type, 
            threshold=thresh
        )
        early_vs_late_pvals[target_label].append(pval)
        
        early_prox = results[f"Early Treg_{target_label}"]['proximity'][len(early_vs_late_pvals[target_label])-1]
        late_prox = results[f"Late Treg_{target_label}"]['proximity'][len(early_vs_late_pvals[target_label])-1]
        sig = "*" if pval < 0.1 else ""
        print(f"{target_label} @ {thresh}px: Early={early_prox:.2%}, Late={late_prox:.2%}, p(Early vs Late)={pval:.4f} {sig}")

# Create side-by-side plots
fig, axes = plt.subplots(1, 2, figsize=(10, 5))

colors = {'Early Treg': '#cca1aa', 'Late Treg': '#68466a'}
markers = {'Early Treg': 'o', 'Late Treg': 's'}

# Left plot: CD9+ Mac
ax = axes[0]
for treg_label in ['Early Treg', 'Late Treg']:
    key = f"{treg_label}_CD9+ Mac"
    proximity_pct = [p*100 for p in results[key]['proximity']]
    
    ax.plot(thresholds, proximity_pct,
            color=colors[treg_label],
            linewidth=2,
            marker=markers[treg_label],
            markersize=8,
            label=treg_label,
            zorder=2)

# Add stars between the lines if Early vs Late is significant
for i, (thresh, pval) in enumerate(zip(thresholds, early_vs_late_pvals['CD9+ Mac'])):
    if pval < 0.1:
        early_val = results['Early Treg_CD9+ Mac']['proximity'][i] * 100
        late_val = results['Late Treg_CD9+ Mac']['proximity'][i] * 100
        ax.text(thresh, early_val+0.75, '*', ha='center', va='center',
               fontsize=20, color='black', zorder=3)

ax.set_xlabel('Distance Threshold (pixels)', fontsize=13)
ax.set_ylabel('% Tregs with CD9+ Mac Neighbor', fontsize=13)
ax.set_title('Proximity to CD9+ Macrophages', fontsize=14)
ax.legend(fontsize=11, frameon=True, loc='best')
ax.grid(True, alpha=0.3, linestyle=':', linewidth=0.8)
ax.set_xticks(thresholds)
ax.tick_params(labelsize=12)

# Right plot: Neutrophil
ax = axes[1]
for treg_label in ['Early Treg', 'Late Treg']:
    key = f"{treg_label}_Neutrophil"
    proximity_pct = [p*100 for p in results[key]['proximity']]
    
    ax.plot(thresholds, proximity_pct,
            color=colors[treg_label],
            linewidth=2,
            marker=markers[treg_label],
            markersize=8,
            label=treg_label,
            zorder=2)

# Add stars between the lines if Early vs Late is significant
for i, (thresh, pval) in enumerate(zip(thresholds, early_vs_late_pvals['Neutrophil'])):
    if pval < 0.1:
        early_val = results['Early Treg_Neutrophil']['proximity'][i] * 100
        late_val = results['Late Treg_Neutrophil']['proximity'][i] * 100
        ax.text(thresh, late_val+0.75, '*', ha='center', va='center',
               fontsize=20, color='black', zorder=3)

ax.set_xlabel('Distance Threshold (pixels)', fontsize=13)
ax.set_ylabel('% Tregs with Neutrophil Neighbor', fontsize=13)
ax.set_title('Proximity to Neutrophils', fontsize=14)
ax.legend(fontsize=11, frameon=True, loc='best')
ax.grid(True, alpha=0.3, linestyle=':', linewidth=0.8)
ax.set_xticks(thresholds)
ax.tick_params(labelsize=12)

plt.tight_layout()
plt.savefig('treg_proximity_sidebyside.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
adata.write_h5ad('/share/fsmresfiles/UC/AtoMx/CD_6k_wholetrans/whole_trans/Processed_merged/h5ad_obj/norm_anno_tregbin.h5ad')

In [ ]:
adata=sc.read_h5ad('/share/fsmresfiles/UC/AtoMx/CD_6k_wholetrans/whole_trans/Processed_merged/h5ad_obj/norm_anno_tregbin.h5ad')